<a href="https://colab.research.google.com/github/chaunijs/onlineshoppingprice/blob/main/vietnam_go.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install playwright deep-translator pandas
!playwright install --with-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.5/47.5 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 3.0 MB/s eta 0:00:00
Installing dependencies...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,915 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy I

In [3]:
from playwright.async_api import async_playwright
import asyncio
import json

async def find_api():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"
        )

        # Capture all API requests
        api_calls = []
        page.on("response", lambda resp: api_calls.append({
            "url": resp.url,
            "status": resp.status,
            "type": resp.request.resource_type
        }))

        await page.goto(
            "https://sieuthi-go.vn/categories/home-care-i.67",
            wait_until="networkidle", timeout=30000
        )
        await asyncio.sleep(3)

        # Dismiss store popup
        try:
            store = await page.wait_for_selector("text=ĐANG CHỌN", timeout=5000)
            if store:
                await store.click()
                await asyncio.sleep(3)
        except:
            pass

        # --- Filter API/XHR calls ---
        print("🔎 API calls found:\n")
        for call in api_calls:
            if call["type"] in ["xhr", "fetch"] and call["status"] == 200:
                url = call["url"]
                # Skip tracking/analytics
                if any(x in url for x in ["google", "facebook", "analytics",
                                           "pixel", "tracking", "fonts",
                                           "gtm", "cdn", ".js", ".css"]):
                    continue
                print(f"   📡 {url[:150]}")

        # --- Try to fetch product API responses ---
        print("\n\n🔎 Looking for product data in API responses...\n")
        for call in api_calls:
            if call["type"] in ["xhr", "fetch"] and call["status"] == 200:
                url = call["url"]
                if any(x in url.lower() for x in ["product", "category", "item",
                                                    "catalog", "api"]):
                    print(f"🎯 Potential product API: {url[:200]}")

                    # Try to get the response body
                    try:
                        # Re-fetch the API
                        resp = await page.request.get(url)
                        body = await resp.json()

                        # Print a sample
                        sample = json.dumps(body, ensure_ascii=False, indent=2)[:2000]
                        print(f"   📦 Response sample:\n{sample}\n")
                    except:
                        print(f"   ⚠️ Could not parse response\n")

        await browser.close()

await find_api()

🔎 API calls found:

   📡 https://wa.onelink.me/v1/onelink
   📡 https://sieuthi-go.vn/api/init
   📡 https://wa.appsflyersdk.com/events?site-id=M7ZR87BU2QFZUh7huS5oK5
   📡 https://wa.onelink.me/v1/onelink?af_id=e4709842-bf4c-4b43-9a75-aa6f39e1f579-p
   📡 https://sieuthi-go.vn/api/store-get-list-store?lang=vi&platform=2
   📡 https://sieuthi-go.vn/api/app-init
   📡 https://sieuthi-go.vn/api/save-store
   📡 https://sieuthi-go.vn/api/loyalty-cart-save?storeId=21
   📡 https://sieuthi-go.vn/api/getSetting
   📡 https://sieuthi-go.vn/api/list-campaign?platform=2&lang=vi
   📡 https://sieuthi-go.vn/api/get-categories?store=102&lang=vi&platform=2
   📡 https://sieuthi-go.vn/api/list-category-quick-ecommerce
   📡 https://sieuthi-go.vn/api/loyalty-cart-view?storeId=21&platform=2&lang=vi
   📡 https://sieuthi-go.vn/api/list-campaign?platform=2&lang=vi
   📡 https://sieuthi-go.vn/api/order2_listProduct?platform=2&lang=vi


🔎 Looking for product data in API responses...

🎯 Potential product API: https://si

In [4]:
from playwright.async_api import async_playwright
from deep_translator import GoogleTranslator
import pandas as pd
import asyncio
import time

BASE_URL = "https://sieuthi-go.vn/categories/home-care-i.67"
all_data = []
seen_names = set()

async def scrape():
    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-dev-shm-usage"]
        )
        page = await browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"
        )

        print(f"\n📄 Loading initial page: {BASE_URL}")
        await page.goto(BASE_URL, wait_until="networkidle", timeout=30000)
        await asyncio.sleep(3)

        # Dismiss store popup ONCE
        try:
            store = await page.wait_for_selector("text=ĐANG CHỌN", timeout=5000)
            if store:
                await store.click()
                await asyncio.sleep(3)
                print("   🏪 Store selected")
        except:
            pass

        page_num = 1

        while page_num <= 20:
            print(f"\n📄 Scraping batch {page_num}...")

            # SMART WAIT: Pause until products are actually present in the DOM
            try:
                await page.wait_for_selector(".stick_camp_prod_item", state="visible", timeout=15000)
            except Exception:
                print("   ⚠️ Timed out waiting for products to load. Stopping.")
                break

            await asyncio.sleep(2)

            # Scrape products
            cards = await page.query_selector_all(".stick_camp_prod_item")
            if not cards:
                print("   ⚠️ No products found. Stopping.")
                break

            before = len(all_data)
            for card in cards:
                name_el = await card.query_selector("p.chakra-text.css-138rv0j")
                name = (await name_el.inner_text()).strip() if name_el else None

                if not name or name in seen_names:
                    continue
                seen_names.add(name)

                sale_el = await card.query_selector("b.chakra-text.css-aqfyqq")
                sale = None
                if sale_el:
                    s = (await sale_el.inner_text()).strip()
                    sale = s.replace("đ", "").replace(",", "").replace(".", "").strip()

                orig_el = await card.query_selector("del.chakra-text.css-nxr2hb")
                original = None
                if orig_el:
                    o = (await orig_el.inner_text()).strip()
                    original = o.replace("đ", "").replace(",", "").replace(".", "").strip()

                disc_el = await card.query_selector("span.p-1")
                discount = (await disc_el.inner_text()).strip() if disc_el else None

                all_data.append({
                    "product_name_vi": name,
                    "sale_price": int(sale) if sale else None,
                    "original_price": int(original) if original else None,
                    "discount_pct": discount,
                })

            print(f"   ✅ +{len(all_data) - before} new (total: {len(all_data)})")

            # Click "See more products" using the corrected class selector
            btn = await page.query_selector("button.ant-btn-round.ant-btn-variant-outlined")
            if not btn or await btn.is_disabled():
                print("🏁 No more 'See more products' button. Done!")
                break

            await btn.scroll_into_view_if_needed()
            await asyncio.sleep(1)

            try:
                # Force click via JavaScript
                await btn.evaluate("node => node.click()")
                print("   🖱️ Clicked 'See more products' (via JavaScript)")

                await asyncio.sleep(5)
                page_num += 1

                # Safeguard
                if len(all_data) == before:
                    print("   ⚠️ Click executed, but no new products appeared. Ending loop.")
                    break
            except Exception as e:
                print(f"🏁 Failed to click 'See more products': {e}. Done!")
                break

        await browser.close()

await scrape()

# --- Result Processing ---
df = pd.DataFrame(all_data)
print(f"\n🎯 Scraped: {len(df)} unique products")

if len(df) > 0:
    print("\n🌐 Translating to English...")
    translator = GoogleTranslator(source="vi", target="en")

    def translate_batch(names, batch_size=20):
        translated = []
        for i in range(0, len(names), batch_size):
            batch = names[i:i+batch_size]
            joined = " ||| ".join(batch)
            try:
                result = translator.translate(joined)
                parts = [p.strip() for p in result.split("|||")]
                if len(parts) == len(batch):
                    translated.extend(parts)
                else:
                    for n in batch:
                        try: translated.append(translator.translate(n))
                        except: translated.append(n)
            except:
                for n in batch:
                    try: translated.append(translator.translate(n))
                    except: translated.append(n)

            print(f"   ✅ {min(i+batch_size, len(names))}/{len(names)}")

        return translated

    df["product_name_en"] = translate_batch(df["product_name_vi"].tolist())
    df = df[["product_name_en", "product_name_vi", "sale_price", "original_price", "discount_pct"]]

    print(f"\n🎯 Final: {len(df)} products")
    print(df[["product_name_en", "sale_price", "original_price"]].head(10).to_string())

    df.to_csv("go_vietnam_home_care.csv", index=False, encoding="utf-8-sig")
    print("\n💾 Saved to go_vietnam_home_care.csv")


📄 Loading initial page: https://sieuthi-go.vn/categories/home-care-i.67
   🏪 Store selected

📄 Scraping batch 1...
   ✅ +54 new (total: 54)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 2...
   ✅ +15 new (total: 69)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 3...
   ✅ +15 new (total: 84)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 4...
   ✅ +15 new (total: 99)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 5...
   ✅ +14 new (total: 113)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 6...
   ✅ +12 new (total: 125)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 7...
   ✅ +11 new (total: 136)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 8...
   ✅ +11 new (total: 147)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping batch 9...
   ✅ +11 new (total: 158)
   🖱️ Clicked 'See more products' (via JavaScript)

📄 Scraping

To Do:
Next Load watchlist